# Homework 2: BERT on AWS

In this homework, we will apply the BERT algorithm on Amazon Web Services.  

This [DataRec repository](https://github.com/sisinflab/DataRec) contains a pointer to accessing recommendation system data, installable via 

```python
pip install datarec-lib
```

**General rules of thumb for homeworks:**
- Read the homework questions carefully.
- Explain your choices.
- Present your findings concisely.
- Use tables, plots, and summary statistics to aid your presentation of findings.
- If you have an idea in mind but could not implement (in code), present the idea thoroughly and how you would have implemented the code. 

### Tasks:

For all tasks below, create one or more functions for each step such that a sequence of functions may be run for a full analysis.  Specify the sequence of functions and their brief descriptions in the README.

1. Download the MovieLens 1m dataset.  You should output a copy of the dataset on an AWS S3 bucket.  
    - Check if your S3 bucket already contains the dataset. If so, the script should not actually download the dataset.

In [2]:
!pip install boto3


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Users\Natha\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
import os
from pathlib import Path
import urllib.request
import zipfile

import boto3
from boto3.s3.transfer import S3UploadFailedError
from botocore.exceptions import ClientError

In [8]:
#bucket_name = 'arn:aws:s3:::stall-de300-winter26'
bucket_name = 'stall-de300-winter26'

In [9]:
s3_client = boto3.client("s3")

In [10]:
sts = boto3.client("sts")
print(sts.get_caller_identity())

{'UserId': 'AIDA5FCD6OFXFLRPMRXVN', 'Account': '904233120110', 'Arn': 'arn:aws:iam::904233120110:user/mycli', 'ResponseMetadata': {'RequestId': '754c1c72-49d7-4122-afd8-f4c5c00afbce', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': '754c1c72-49d7-4122-afd8-f4c5c00afbce', 'x-amz-sts-extended-request-id': 'MTp1cy1lYXN0LTI6UzoxNzcxMTA2NzczMTEzOlI6UWNmT3J2d28=', 'content-type': 'text/xml', 'content-length': '402', 'date': 'Sat, 14 Feb 2026 22:06:13 GMT'}, 'RetryAttempts': 0}}


In [11]:
MOVIELENS_1M_URL = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"
MOVIELENS_DIR = Path("ml-1m")
MOVIELENS_ZIP = Path("ml-1m.zip")


def _get_bucket_name(arn_or_name: str) -> str:
    """Accept either a plain bucket name or an S3 bucket ARN and
    return the bucket name portion that the boto3 S3 client expects."""
    prefix = "arn:aws:s3:::"
    if arn_or_name.startswith(prefix):
        return arn_or_name[len(prefix) :]
    return arn_or_name


def download_movielens_1m() -> Path:
    """Download the MovieLens 1M dataset if it is not already present locally.

    Returns
    -------
    Path
        Path to the extracted MovieLens 1M directory.
    """
    # If the directory already exists and is non-empty, assume it's usable.
    if MOVIELENS_DIR.exists() and any(MOVIELENS_DIR.iterdir()):
        print(f"MovieLens 1M already present at {MOVIELENS_DIR.resolve()}")
        return MOVIELENS_DIR

    # Ensure we have the ZIP file; download only if missing.
    if not MOVIELENS_ZIP.exists():
        print(f"Downloading MovieLens 1M from {MOVIELENS_1M_URL} ...")
        try:
            urllib.request.urlretrieve(MOVIELENS_1M_URL, MOVIELENS_ZIP)
        except Exception as exc:
            raise RuntimeError(f"Failed to download MovieLens 1M: {exc}") from exc

    # Extract the archive (creates 'ml-1m' directory).
    print(f"Extracting {MOVIELENS_ZIP} ...")
    try:
        with zipfile.ZipFile(MOVIELENS_ZIP, "r") as zf:
            zf.extractall()
    except zipfile.BadZipFile as exc:
        raise RuntimeError(f"Corrupted MovieLens ZIP file: {exc}") from exc

    if not MOVIELENS_DIR.exists():
        raise FileNotFoundError(
            f"Expected directory {MOVIELENS_DIR} not found after extraction."
        )

    print(f"MovieLens 1M extracted to {MOVIELENS_DIR.resolve()}")
    return MOVIELENS_DIR


def s3_bucket_contains() -> bool:
    """Check whether the configured S3 bucket already contains a copy
    of the MovieLens 1M dataset under the 'ml-1m/' prefix."""
    bucket = _get_bucket_name(bucket_name)
    try:
        response = s3_client.list_objects_v2(Bucket=bucket, Prefix="ml-1m/")
    except ClientError as exc:
        # If we cannot list objects (e.g., permissions), treat as empty so
        # that the rest of the pipeline can attempt an upload.
        print(f"Warning: failed to list objects in bucket {bucket}: {exc}")
        return False

    has_objects = "Contents" in response and len(response["Contents"]) > 0
    if has_objects:
        print(f"S3 bucket '{bucket}' already contains MovieLens 1M data under 'ml-1m/'.")
    else:
        print(f"S3 bucket '{bucket}' does not yet contain MovieLens 1M data.")
    return has_objects


def send_to_s3_bucket():
    """Upload the local MovieLens 1M dataset to the configured S3 bucket
    under the 'ml-1m/' prefix. Assumes the dataset has already been
    downloaded locally."""
    bucket = _get_bucket_name(bucket_name)

    if not MOVIELENS_DIR.exists() or not any(MOVIELENS_DIR.iterdir()):
        raise FileNotFoundError(
            f"Local MovieLens directory '{MOVIELENS_DIR}' does not exist or is empty. "
            "Run download_movielens_1m() first."
        )

    print(f"Uploading contents of {MOVIELENS_DIR.resolve()} to s3://{bucket}/ml-1m/ ...")
    for local_path in MOVIELENS_DIR.rglob("*"):
        if not local_path.is_file():
            continue

        # Construct an S3 key that preserves the relative directory structure.
        relative_path = local_path.relative_to(MOVIELENS_DIR)
        s3_key = f"ml-1m/{relative_path.as_posix()}"

        try:
            s3_client.upload_file(
                Filename=str(local_path),
                Bucket=bucket,
                Key=s3_key,
            )
            print(f"Uploaded {local_path} -> s3://{bucket}/{s3_key}")
        except (S3UploadFailedError, ClientError) as exc:
            print(f"Failed to upload {local_path} to s3://{bucket}/{s3_key}: {exc}")
            # Depending on requirements, you could choose to raise here.
            # For now, we log and continue with other files.

In [12]:
def orchestrate_movielens_download():
    if s3_bucket_contains():
        return
    
    download_movielens_1m()
    send_to_s3_bucket()

In [13]:
orchestrate_movielens_download()

MovieLens 1M already present at C:\Users\Natha\source\repos\Classes\DE_300\homework\homework_2\ml-1m
Uploading contents of C:\Users\Natha\source\repos\Classes\DE_300\homework\homework_2\ml-1m to s3://stall-de300-winter26/ml-1m/ ...
Failed to upload ml-1m\movies.dat to s3://stall-de300-winter26/ml-1m/movies.dat: Failed to upload ml-1m\movies.dat to stall-de300-winter26/ml-1m/movies.dat: An error occurred (AccessDenied) when calling the PutObject operation: Access Denied
Failed to upload ml-1m\ratings.dat to s3://stall-de300-winter26/ml-1m/ratings.dat: Failed to upload ml-1m\ratings.dat to stall-de300-winter26/ml-1m/ratings.dat: An error occurred (AccessDenied) when calling the CreateMultipartUpload operation: Access Denied
Failed to upload ml-1m\README to s3://stall-de300-winter26/ml-1m/README: Failed to upload ml-1m\README to stall-de300-winter26/ml-1m/README: An error occurred (AccessDenied) when calling the PutObject operation: Access Denied
Failed to upload ml-1m\users.dat to s3://s

2. Create embeddings for the BERT algorithm.  For the same set of items (movies) in the dataset, you should create the embeddings once and output a copy of the necessary intermediate results on the S3 bucket.
    - This is the *offline* step, where embeddings only need to be created once for the recommendation system.
    - Use a random subset (30%) of users in the available dataset.

3. Recommend five movies for each of the following users.  The recommendations should be saved in a file on the S3 bucket containing `User_Type`, `Last_Interaction_Time`, other user summaries in the dataset,and a list of recommended movies:
    - *Cold user*: a user that the system has no data on.
    - *Top user*: a random user who has frequently rated movies (number of interactions among the top 5\% of users).

4. Repeat steps 2 and 3 but with the full set of data.  You should be able to reuse your work from earlier.

5. Choose and rate 10 movies and create a "user profile" for yourself.  Save your user profile on the S3 bucket.  Recommend 5 movies for yourself and save the results on the S3 bucket.

# Submission guidelines
Your submission should be contained in a `homework_2` folder of your Github repository, and it should include 
- a `readme.md` file including how to run the code and what your expected outputs are (if the code is run), 
- your source code, and/or
- a `.pdf` or `.html` file containing any necessary observations and details.
    - If you find your source code self-explanatory, you may opt to skip the `.pdf` or `.html` file in this homework.


# Generative AI disclosure

*Syllabus* policy: 

Required disclosure: each submission must include an AI Usage note stating: (1) tool(s) used, (2) the key prompt(s), and (3) what you changed and how you verified the results. If none, write: “AI Usage: None.”